#### **DADOS DO TESOURO NACIONAL E SICONFI**

**RESUMO**  
Apresenta o roteiro direto para baixar os dados públicos do Tesouro Transparente e do SICONFI para os estados brasileiros entre 2013 e 2025. Os dados são brutos, em valores nominais, conforme disponibilizados pelo SICONFI.

**Obs.:** disponível em <https://tesourotransparente.gov.br/consultas/consultas-siconfi/siconfi-api-de-dados-abertos>

---------------------------------------------------
**AUTOR**: Prof. Dr. Elson Rodrigo de Souza Santos  
**E-MAIL**: [elson129@gmail.com] ou [elson.rodrigo@ufabc.edu.br]  
**REPOSITÓRIO**: https://github.com/elson29r/model_data_general  
**DATA**: Agosto de 2026

**Sumário**
1) Instalação dos pacotes
2) Extração e exportação dos dados fiscais
3) Filtro e tratamento

**1) Instalação dos pacotes:**  

Pacotes utilizados:

- **Pandas** - análise e manutenção de em Python. O pacote fornece estruturas de dados eficientes, chamadas DataFrames, ideais para organizar painéis longitudinais, tratar séries temporais e preparar a matriz final de variáveis;

- **Requests** - biblioteca Python para realizar requisições HTTP.

In [ ]:
# Instalação

!pip install pandas requests

**2) Extração e exportação dos dados fiscais**  

Extraímos e exportamos os dados fiscais brutos diretamente do API do Tesouro Transparente entre 2013 e 2025.

**Importante!!!!** Vamos baixar a Declaração de Contas Anuais (DCA) em que apresenta uma grande quantidade de linhas e informações sobre cada estado. Automatizamos para exportação para CSV

In [ ]:
# Declaração de Contas Anuais - DCA

# siconfi_dca.py

from pathlib import Path
from typing import Iterable

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca"

ESTADOS_IBGE = {
    "11": "Rondônia", "12": "Acre", "13": "Amazonas", "14": "Roraima",
    "15": "Pará", "16": "Amapá", "17": "Tocantins", "21": "Maranhão",
    "22": "Piauí", "23": "Ceará", "24": "Rio Grande do Norte", "25": "Paraíba",
    "26": "Pernambuco", "27": "Alagoas", "28": "Sergipe", "29": "Bahia",
    "31": "Minas Gerais", "32": "Espírito Santo", "33": "Rio de Janeiro",
    "35": "São Paulo", "41": "Paraná", "42": "Santa Catarina",
    "43": "Rio Grande do Sul", "50": "Mato Grosso do Sul", "51": "Mato Grosso",
    "52": "Goiás", "53": "Distrito Federal",
}


def criar_sessao() -> requests.Session:
   
    sessao = requests.Session()
    retentativas = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    sessao.mount("https://", HTTPAdapter(max_retries=retentativas))
    return sessao


def consultar_dca(sessao: requests.Session, id_ente: str, ano: int) -> pd.DataFrame:
   
    registros = []
    offset = 0

    while True:
        params = {
            "an_exercicio": ano,
            "id_ente": id_ente,
            "offset": offset,
        }
        resposta = sessao.get(BASE_URL, params=params, timeout=60)
        resposta.raise_for_status()
        dados = resposta.json()

        itens = dados.get("items", [])
        registros.extend(itens)

        if not dados.get("hasMore", False):
            break
        offset += len(itens)

    return pd.DataFrame(registros)


def get_dca_estados(
    ids_estados: Iterable[str] = ESTADOS_IBGE.keys(),
    first_year: int = 2013,
    last_year: int = 2025,
    output_folder: str = "siconfi_dca",
) -> pd.DataFrame:
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    sessao = criar_sessao()
    partes = []

    for ano in range(first_year, last_year + 1):
        for id_ente in ids_estados:
            try:
                df = consultar_dca(sessao, id_ente, ano)
            except requests.RequestException as erro:
                print(f"Falha em {ano}/{ESTADOS_IBGE.get(id_ente, id_ente)}: {erro}")
                continue

            if df.empty:
                print(f"{ano} {ESTADOS_IBGE.get(id_ente, id_ente)}: sem dados.")
                continue

            partes.append(df)
            print(f"{ano} {ESTADOS_IBGE.get(id_ente, id_ente)}: {df.shape[0]} linhas")

    if not partes:
        print("Nenhum dado retornado para o recorte informado.")
        return pd.DataFrame()

    resultado = pd.concat(partes, ignore_index=True)
    resultado.to_csv(output_path / "dca_estados.csv", index=False)
    return resultado


if __name__ == "__main__":
    dca = get_dca_estados(first_year=2013, last_year=2025)
    print(dca.shape)

In [ ]:
# Verificação

# dimensões e primeiras linhas
print(dca.shape)
print(dca.head())

# tipos de dado por coluna, útil para conferir se algo que deveria
# ser numérico veio como texto
print(dca.dtypes)

# valores ausentes por coluna
print(dca.isna().sum())

# cobertura de anos e entes efetivamente presentes no resultado
print(sorted(dca["exercicio"].unique()))
print(sorted(dca["cod_ibge"].unique()))

# contagem de linhas por ano, ajuda a identificar se algum ano
# veio incompleto ou vazio
print(dca.groupby("exercicio").size())

# checagem rápida de duplicidade
print(dca.duplicated().sum())

**3) Filtro e tratamento**

Devido a dimensão do dataframe (924.499 linhas x 11 colunas) é pertinente filtrar as informações alvo.

Para o projeto consideramos as variáveis de interesse:  
- Receita total  
- Receita corrente  
- Receita de capital   
- Operações de cŕedito  
- Despesa total  
- Despesa primária  
- Despesa corrente  
- Despesa juros e encargos  
- Despesas de capital  
- Investimento  
- Amortizações     
- Ativo (financeiro e não financeiro)
- Passivo (financeiro e não financeiro)
- Patrimônio Líquido



**Nota**: as informaçoes sobre cada estado são extremamente ricas e detalhas. Obviamente que pode ser usado o mesmo procedimento para escolher outras variáveis.


**Sumário**  
3.1 Separação e padronização das informações    
3.2 Identificar as variáveis de interesse  
3.3 Filtrar as variáveis de interesse e criar uma nova base
 

In [ ]:
# 3.1 Separação e padronização das informações  

# Identificação das colunas

print(dca.columns.tolist())

In [ ]:
# valores únicos da coluna 'anexo'

print(dca["anexo"].unique())

In [ ]:
# normaliza o rótulo do anexo antes de agrupar

dca["anexo_normalizado"] = dca["anexo"].str.replace("^DCA-", "", regex=True)

print(dca["anexo_normalizado"].unique())

In [ ]:
print(dca.groupby(["exercicio", "anexo"]).size().unstack(fill_value=0))

In [ ]:
dca_por_anexo = {
    anexo: grupo.reset_index(drop=True)
    for anexo, grupo in dca.groupby("anexo_normalizado")
}

for anexo, df in dca_por_anexo.items():
    print(f"{anexo}: {df.shape[0]} linhas")

In [ ]:
from pathlib import Path

pasta_saida = Path("siconfi_dca_por_anexo")
pasta_saida.mkdir(parents=True, exist_ok=True)

nomes_arquivo = {
    "Anexo I-AB": "balanco_patrimonial",
    "Anexo I-C": "receitas_orcamentarias",
    "Anexo I-D": "despesas_orcamentarias",
    "Anexo I-E": "despesas_por_funcao",
    "Anexo I-F": "restos_a_pagar_despesas",
    "Anexo I-G": "restos_a_pagar_despesas_por_funcao",
    "Anexo I-HI": "variacoes_patrimoniais",
}

for anexo, df in dca_por_anexo.items():
    nome_arquivo = nomes_arquivo.get(anexo, anexo.replace(" ", "_").replace("/", "-"))
    df.to_csv(pasta_saida / f"{nome_arquivo}.csv", index=False)
    print(f"{anexo}: {df.shape[0]} linhas exportadas para {nome_arquivo}.csv")

In [ ]:
# 3.2 Identificar as variáveis de interesse

# identificar a receita total em 'conta' e 'cod_conta'

receitas = dca_por_anexo["Anexo I-C"]
despesas = dca_por_anexo["Anexo I-D"]
patrimonial = dca_por_anexo["Anexo I-AB"]

# procurar os códigos

print(sorted(despesas["cod_conta"].astype(str).str.strip().unique()))
print(sorted(receitas["cod_conta"].astype(str).str.strip().unique()))
print(sorted(patrimonial["cod_conta"].astype(str).str.strip().unique()))

In [ ]:
# 3.3 Filtrar as variáveis de interesse e criar uma nova base

import re

# --- mapeamento de código para nome padronizado, por anexo ---

mapa_receita = {
    "TotalReceitas": "rec_total",
    "RO1.0.0.0.00.00.00": "rec_cor",
    "RO2.0.0.0.00.00.00": "rec_cap",
    "RO2.1.0.0.00.00.00": "rec_cap_cred",
}

mapa_despesa = {
    "TotalDespesas": "desp_total",
    "DO3.0.00.00.00.00": "desp_cor",
    "DO3.2.00.00.00.00": "desp_juros",
    "DO4.0.00.00.00.00": "desp_cap",
    "DO4.4.00.00.00.00": "desp_cap_invest",
    "DO4.6.00.00.00.00": "desp_cap_amort",
}

mapa_balanco_patrimonial = {
    "Ativo": "ati",
    "AtivoFinanceiro": "ati_fin",
    "AtivoPermanente": "ati_nfin",  # nomenclatura pré-reforma MCASP; confirmar equivalência pós-2013
    "AtivoReal": "ati_real",
    "Passivo": "pas",
    "PassivoFinanceiro": "pas_fin",
    "PassivoPermanente": "pas_nfin",  # idem
    "PassivoReal": "pas_real",
    "PatrimonioLiquido": "patri_liq",
    # pas_findep, pas_fin_rest: sem correspondência identificada na lista de códigos
}


def normalizar_codigo(serie_cod_conta: "pd.Series") -> "pd.Series":
    """Padroniza cod_conta como string, sem espaços nas pontas,
    para casar de forma consistente entre anos e anexos."""
    return serie_cod_conta.astype(str).str.strip()


def filtrar_variaveis(df: "pd.DataFrame", mapa_codigos: dict) -> "pd.DataFrame":
    """Filtra e rotula as linhas de um anexo conforme o mapa de código
    para nome padronizado, casando por cod_conta."""
    df = df.copy()
    df["cod_conta_normalizado"] = normalizar_codigo(df["cod_conta"])
    df["variavel"] = df["cod_conta_normalizado"].map(mapa_codigos)

    filtrado = df[df["variavel"].notna()].copy()

    print(f"{filtrado.shape[0]} linhas casadas com o mapa de códigos.")
    print(f"Variáveis encontradas: {sorted(filtrado['variavel'].unique())}")

    codigos_sem_correspondencia = set(mapa_codigos) - set(filtrado["variavel"].map({v: k for k, v in mapa_codigos.items()}))
    if codigos_sem_correspondencia:
        print(f"Códigos do mapa sem nenhuma linha correspondente: {codigos_sem_correspondencia}")

    return filtrado


def pivotar_variaveis(df_filtrado: "pd.DataFrame") -> "pd.DataFrame":
    """Converte o formato longo filtrado em formato largo, uma coluna
    por variável, indexado por exercício e ente."""
    return df_filtrado.pivot_table(
        index=["exercicio", "cod_ibge", "uf"],
        columns="variavel",
        values="valor",
        aggfunc="first",
    ).reset_index()


# --- aplicação: cada anexo filtrado e pivotado separadamente ---

receitas = dca_por_anexo["Anexo I-C"]
despesas = dca_por_anexo["Anexo I-D"]
patrimonial = dca_por_anexo["Anexo I-AB"]

rec_filtrado = filtrar_variaveis(receitas, mapa_receita)
desp_filtrado = filtrar_variaveis(despesas, mapa_despesa)
bp_filtrado = filtrar_variaveis(patrimonial, mapa_balanco_patrimonial)

base_rec = pivotar_variaveis(rec_filtrado)
base_desp = pivotar_variaveis(desp_filtrado)
base_bp = pivotar_variaveis(bp_filtrado)

# junta receita e despesa antes de calcular as variáveis construídas
base_rd = base_rec.merge(base_desp, on=["exercicio", "cod_ibge", "uf"], how="outer")

base_rd["desp_pri"] = base_rd["desp_total"] - base_rd["desp_juros"] - base_rd["desp_cap_amort"]
base_rd["desp_corr"] = base_rd["desp_cor"] - base_rd["desp_juros"]
base_rd["rec_cap_out"] = base_rd["rec_cap"] - base_rd["rec_cap_cred"]

base_final = base_rd.merge(base_bp, on=["exercicio", "cod_ibge", "uf"], how="outer")
base_final.to_csv("dca_variaveis_selecionadas.csv", index=False)

#### **FIM**